### Sigmoid Türevinin Adım Adım İspatı

Sigmoid fonksiyonu: $\sigma(x) = \frac{1}{1+e^{-x}} = (1+e^{-x})^{-1}$

**1. Adım: Zincir kuralını uygulayalım:**
$$\frac{d}{dx} \sigma(x) = -1 \cdot (1+e^{-x})^{-2} \cdot \frac{d}{dx}(1+e^{-x})$$

**2. Adım: İçteki ifadenin türevini alalım:**
$$\frac{d}{dx}(1+e^{-x}) = -e^{-x}$$

**3. Adım: İfadeleri birleştirelim:**
$$\frac{d}{dx} \sigma(x) = -(1+e^{-x})^{-2} \cdot (-e^{-x}) = \frac{e^{-x}}{(1+e^{-x})^2}$$

**4. Adım: İfadeyi $\sigma(x)$ cinsinden yazmak için parçalayalım:**
$$\frac{d}{dx} \sigma(x) = \frac{1}{1+e^{-x}} \cdot \frac{e^{-x}}{1+e^{-x}}$$

**5. Adım: Pay kısmına 1 ekleyip çıkaralım:**
$$\frac{e^{-x}}{1+e^{-x}} = \frac{1+e^{-x}-1}{1+e^{-x}} = \frac{1+e^{-x}}{1+e^{-x}} - \frac{1}{1+e^{-x}} = 1 - \sigma(x)$$

**Sonuç:**
$$\frac{d}{dx} \sigma(x) = \sigma(x) (1 - \sigma(x))$$

### Ağırlık Parametresi (w) için Zincir Kuralı

Bir yapay sinir ağında genellikle $z = wx + b$ ve $a = \sigma(z)$ şeklindedir. MSE kaybının ağırlık ($w$) parametresine göre türevi (gradyanı) zincir kuralı ile şöyle hesaplanır:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$$

Bileşenleri tek tek hesaplayalım:
1.  **Kayıp Türevi:** $\frac{\partial L}{\partial a} = (a - y)$
2.  **Aktivasyon Türevi:** $\frac{\partial a}{\partial z} = \sigma(z)(1 - \sigma(z)) = a(1 - a)$
3.  **Girdi Türevi:** $\frac{\partial z}{\partial w} = x$

**Tam Formül:**
$$\frac{\partial L}{\partial w} = (a - y) \cdot a(1 - a) \cdot x$$

Bu ifade, geri yayılım sırasında ağırlığın ne yönde güncelleneceğini belirler. Görüldüğü gibi, gradyan hem tahmin hatasına ($a-y$), hem sigmoid türevine, hem de girdinin ($x$) değerine bağlıdır.

In [18]:
import torch

# Değerler
w1 = torch.tensor(3.0, requires_grad=True)
x1 = torch.tensor(1.0, requires_grad=True)

def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

# === İLERİ YAYILIM (Forward Pass) ===
print("=== İLERİ YAYILIM ===")
y1 = w1 * x1
print(f"y1 = w1 * x1 = {w1.item()} * {x1.item()} = {y1.item()}")

a1 = sigmoid(y1)
print(f"a1 = sigmoid(y1) = sigmoid({y1.item():.4f}) = {a1.item():.4f}")

target = 2.0
loss = 0.5 * (a1 - target) ** 2
print(f"loss = 0.5 * (a1 - {target})^2 = 0.5 * ({a1.item():.4f} - {target})^2 = {loss.item():.4f}")

# === GERİYE YAYILIM (PyTorch Autograd) ===
print("\n=== GERİYE YAYILIM (PyTorch Autograd) ===")
loss.backward()
print(f"w1.grad (PyTorch) = {w1.grad.item():.6f}")
print(f"x1.grad (PyTorch) = {x1.grad.item():.6f}")

# === MANUEL GERİYE YAYILIM (Zincir Kuralı) ===
print("\n=== MANUEL GERİYE YAYILIM (Adım Adım) ===")

# Adım 1: dL/da1
dL_da1 = a1.item() - target
print(f"1. dL/da1 = a1 - target = {a1.item():.4f} - {target} = {dL_da1:.6f}")

# Adım 2: da1/dy1 = sigmoid'(y1)
sigmoid_y1 = sigmoid(y1).item()
da1_dy1 = sigmoid_y1 * (1 - sigmoid_y1)
print(f"2. da1/dy1 = sigmoid(y1)*(1-sigmoid(y1)) = {sigmoid_y1:.4f} * {1-sigmoid_y1:.4f} = {da1_dy1:.6f}")

# Adım 3: dL/dy1 = dL/da1 * da1/dy1
dL_dy1 = dL_da1 * da1_dy1
print(f"3. dL/dy1 = dL/da1 * da1/dy1 = {dL_da1:.6f} * {da1_dy1:.6f} = {dL_dy1:.6f}")

# Adım 4: dy1/dw1 = x1
dy1_dw1 = x1.item()
print(f"4. dy1/dw1 = x1 = {dy1_dw1}")

# Adım 5: dL/dw1 = dL/dy1 * dy1/dw1
dL_dw1 = dL_dy1 * dy1_dw1
print(f"5. dL/dw1 = dL/dy1 * dy1/dw1 = {dL_dy1:.6f} * {dy1_dw1} = {dL_dw1:.6f}")

# Adım 6: dy1/dx1 = w1
dy1_dx1 = w1.item()
print(f"6. dy1/dx1 = w1 = {dy1_dx1}")

# Adım 7: dL/dx1 = dL/dy1 * dy1/dx1
dL_dx1 = dL_dy1 * dy1_dx1
print(f"7. dL/dx1 = dL/dy1 * dy1/dx1 = {dL_dy1:.6f} * {dy1_dx1} = {dL_dx1:.6f}")

print("\n=== KARŞILAŞTIRMA ===")
print(f"w1 grad → Manuel: {dL_dw1:.6f} | PyTorch: {w1.grad.item():.6f}")
print(f"x1 grad → Manuel: {dL_dx1:.6f} | PyTorch: {x1.grad.item():.6f}")

=== İLERİ YAYILIM ===
y1 = w1 * x1 = 3.0 * 1.0 = 3.0
a1 = sigmoid(y1) = sigmoid(3.0000) = 0.9526
loss = 0.5 * (a1 - 2.0)^2 = 0.5 * (0.9526 - 2.0)^2 = 0.5486

=== GERİYE YAYILIM (PyTorch Autograd) ===
w1.grad (PyTorch) = -0.047319
x1.grad (PyTorch) = -0.141958

=== MANUEL GERİYE YAYILIM (Adım Adım) ===
1. dL/da1 = a1 - target = 0.9526 - 2.0 = -1.047426
2. da1/dy1 = sigmoid(y1)*(1-sigmoid(y1)) = 0.9526 * 0.0474 = 0.045177
3. dL/dy1 = dL/da1 * da1/dy1 = -1.047426 * 0.045177 = -0.047319
4. dy1/dw1 = x1 = 1.0
5. dL/dw1 = dL/dy1 * dy1/dw1 = -0.047319 * 1.0 = -0.047319
6. dy1/dx1 = w1 = 3.0
7. dL/dx1 = dL/dy1 * dy1/dx1 = -0.047319 * 3.0 = -0.141958

=== KARŞILAŞTIRMA ===
w1 grad → Manuel: -0.047319 | PyTorch: -0.047319
x1 grad → Manuel: -0.141958 | PyTorch: -0.141958
